# TIDA — Experiment Runner (Colab)

Mounts Google Drive for persistent storage, installs dependencies, and runs the TIDA experiment suite.
Results and checkpoints survive session restarts via Drive.

**Runtime → Change runtime type → A100 GPU** for fastest training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/tida_experiments"
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists("/content/tida"):
    !ln -s "$DRIVE_DIR" /content/tida
os.chdir("/content/tida")

In [ ]:
# Clone repo (first run only)
if not os.path.exists("/content/tida/tida_project"):
    !git clone https://github.com/shervinemp/TIDA.git /content/_tida_temp
    !shopt -s dotglob; mv /content/_tida_temp/* /content/tida/; rmdir /content/_tida_temp

os.chdir("/content/tida/tida_project")

In [ ]:
# Install dependencies
# Colab ships torchao which can conflict; we don't use it
!pip uninstall -y torchao 2>/dev/null || true
!pip install --quiet torch transformers accelerate peft datasets tqdm
!pip install --quiet --upgrade huggingface-hub

In [ ]:
# Verify GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Preview
!python experiments.py --dry-run

In [ ]:
# @title Run experiments
# @markdown Edit the filter and seed count, then run this cell.

FILTER = "_tiny"   # @param {"type":"string"} "" for all, "_tiny" for TinyLlama, "_small" for Phi-3
SEEDS = 3           # @param {"type":"integer"} Number of seeds per experiment

if FILTER:
    !python experiments.py --name "$FILTER" --seeds "$SEEDS"
else:
    !python experiments.py --seeds "$SEEDS"

print("\nDone. Results saved to Drive:")
print(f"  {DRIVE_DIR}/tida_project/results/experiments.json")
print(f"  {DRIVE_DIR}/tida_project/results/samples/")

In [ ]:
# Quick results summary
import json
results_path = "results/experiments.json"
if os.path.exists(results_path):
    print("Completed experiments:")
    !python experiments.py --list
else:
    print("No results yet.")

---
**Notes:**
- TinyLlama experiments take ~2–3 hours each on A100 (7 × 3 = ~2 days for all TinyLlama).
- Results are saved incrementally — if the session disconnects, partial results are preserved.
- To resume, re-run cells 1-3, then re-run the experiment cell with `--force` to skip completed runs.
- Intermediate checkpoints are automatically cleaned up — only the final epoch is kept.